In [1]:
!pip install librosa

In [2]:
import numpy as np
import pandas as pd
import os
import librosa
import librosa.display
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [3]:
# dataset/
#    rock/
#    pop/
#    jazz/
#    classical/
#    hiphop/
#    electronic/
!wget https://web.archive.org/web/20220328223413/ftp://opihi.cs.uvic.ca/sound/genres.tar.gz
!tar -xzf genres.tar.gz
DATASET_PATH = "/content/genres"
print("--->",DATASET_PATH)
print("--->",os.listdir(DATASET_PATH))

SAMPLE_RATE = 22050
DURATION = 30  # segundos
SAMPLES_PER_TRACK = SAMPLE_RATE * DURATION

--2026-03-30 00:42:16--  https://web.archive.org/web/20220328223413/ftp://opihi.cs.uvic.ca/sound/genres.tar.gz
Resolving web.archive.org (web.archive.org)... 207.241.237.3
Connecting to web.archive.org (web.archive.org)|207.241.237.3|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1225571541 (1.1G) [application/x-gzip]
Saving to: ‘genres.tar.gz’

genres.tar.gz       100%[===================>]   1.14G  9.99MB/s    in 2m 24s  

2026-03-30 00:45:06 (8.11 MB/s) - ‘genres.tar.gz’ saved [1225571541/1225571541]

---> /content/genres
---> ['di.mf', 'country', 'ro.mf', 'rock', 'jazz', 'hi.mf', 'classical', 'ja.mf', 'blues', 'cl.mf', 'bextract_single.mf', 'disco', 'pop', 're.mf', 'me.mf', 'bl.mf', 'hiphop', 'input.mf', 'po.mf', 'metal', 'co.mf', 'reggae']


In [4]:
def extract_features(dataset_path, n_mfcc=13, duration=30):
    X = []
    y = []

    SAMPLES_PER_TRACK = SAMPLE_RATE * duration
    VALID_EXTENSIONS = (".wav", ".au", ".mp3")

    for genre in os.listdir(dataset_path):
        genre_path = os.path.join(dataset_path, genre)

        if not os.path.isdir(genre_path):
            continue

        print(f"Procesando: {genre}")

        for file in os.listdir(genre_path):
            file_path = os.path.join(genre_path, file)

            if not file.lower().endswith(VALID_EXTENSIONS):
                continue

            try:
                signal, sr = librosa.load(file_path, sr=SAMPLE_RATE)

                # 🔥 Forzar misma duración
                if len(signal) >= SAMPLES_PER_TRACK:
                    signal = signal[:SAMPLES_PER_TRACK]
                else:
                    padding = SAMPLES_PER_TRACK - len(signal)
                    signal = np.pad(signal, (0, padding))

                mfcc = librosa.feature.mfcc(
                    y=signal,
                    sr=sr,
                    n_mfcc=n_mfcc
                )
                mfcc = mfcc.T

                X.append(mfcc)
                y.append(genre)

            except Exception as e:
                print(f"Error en {file_path}: {e}")

    return np.array(X), np.array(y)

In [5]:
X, y = extract_features(DATASET_PATH)

print(f"Total muestras: {len(X)}")

Procesando: country
Procesando: rock
Procesando: jazz
Procesando: classical
Procesando: blues
Procesando: disco
Procesando: pop
Procesando: hiphop
Procesando: metal
Procesando: reggae
Total muestras: 1000


In [6]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LEN = 1300  # ajustable

X_padded = pad_sequences(X, maxlen=MAX_LEN, padding='post', truncating='post')

In [7]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_categorical = to_categorical(y_encoded)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X_padded, y_categorical, test_size=0.2, random_state=42
)

In [9]:
model = Sequential()

model.add(LSTM(128, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dropout(0.3))

model.add(Dense(64, activation='relu'))
model.add(Dense(y_categorical.shape[1], activation='softmax'))

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128)            │        72,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 81,610 (318.79 KB)

 Trainable params: 81,610 (318.79 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=150,
    batch_size=32
)

Epoch 1/150
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.9887 - loss: 0.0354 - val_accuracy: 0.5700 - val_loss: 2.5230
Epoch 2/150
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - accuracy: 0.9875 - loss: 0.0423 - val_accuracy: 0.5750 - val_loss: 2.5273
Epoch 3/150
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 0.9950 - loss: 0.0253 - val_accuracy: 0.5900 - val_loss: 2.5862
Epoch 4/150
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 0.9937 - loss: 0.0218 - val_accuracy: 0.6050 - val_loss: 2.5502
Epoch 5/150
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - accuracy: 0.9900 - loss: 0.0297 - val_accuracy: 0.5850 - val_loss: 2.6640
Epoch 6/150
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 0.9912 - loss: 0.0237 - val_accuracy: 0.6250 - val_loss: 2.4719
Epoch 7/150
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 73ms/step - accuracy: 0.9925 - loss: 0.0288 - val_accuracy: 0.5850 - val_loss: 2.7340
Epoch 8/150
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 58ms/step - accuracy: 0.9638 - loss: 0.1033 - val_accuracy: 0.

In [19]:
loss, acc = model.evaluate(X_test, y_test)
print(f"Accuracy: {acc:.2f}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.6050 - loss: 3.6083
Accuracy: 0.61


In [20]:
def predict(file_path, model, label_encoder):
    signal, sr = librosa.load(file_path, sr=SAMPLE_RATE)
    mfcc = librosa.feature.mfcc(
        y=signal,
        sr=sr,
        n_mfcc=13
    )
    mfcc = mfcc.T

    mfcc = pad_sequences([mfcc], maxlen=MAX_LEN, padding='post', truncating='post')

    prediction = model.predict(mfcc)
    predicted_label = label_encoder.inverse_transform([np.argmax(prediction)])

    return predicted_label[0]

In [21]:
test_file = "/content/test.wav"
print(predict(test_file, model, le))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
country


In [22]:
model.save("music_genre_classifier.h5")